# Notebook 08 — Feature Engineering

This notebook engineers features based on crop physiology, including multi-month rolling rainfall, weather anomalies, soil moisture lags (representing water tables), and meteorological interactions.

## 1. Import Packages & Local Configuration

In [9]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure project root is in the path
sys.path.append(str(Path(os.getcwd()).parent))

import config
print("Features Output Path:", config.PROCESSED_DIR / "master_features.csv")

SyntaxError: invalid syntax (3262073054.py, line 3)

## 2. Load Master Dataset

In [ ]:
df = pd.read_csv(config.PROCESSED_DIR / "master_dataset.csv")
df = df.sort_values(by=["District", "Year", "Month"]).reset_index(drop=True)
print("Master dataset columns:", list(df.columns))

## 3. Rolling Averages

In [ ]:
df["Rainfall_Roll_3m"] = df.groupby("District")["Rainfall"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df["Rainfall_Roll_6m"] = df.groupby("District")["Rainfall"].transform(lambda x: x.rolling(6, min_periods=1).mean())
df["Rainfall_Roll_12m"] = df.groupby("District")["Rainfall"].transform(lambda x: x.rolling(12, min_periods=1).mean())

df["Soil_Moisture_Root_Roll_3m"] = df.groupby("District")["Soil_Moisture_Root"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df["Soil_Moisture_Root_Roll_6m"] = df.groupby("District")["Soil_Moisture_Root"].transform(lambda x: x.rolling(6, min_periods=1).mean())

print(df[["District", "Year", "Month", "Rainfall", "Rainfall_Roll_3m", "Soil_Moisture_Root_Roll_3m"]].head())

## 4. Climatological Anomalies

In [ ]:
# Calculate monthly climatological means per district & month
climatology = df.groupby(["District", "Month"])[["Rainfall", "Temperature", "Soil_Moisture_Root"]].transform("mean")
climatology.columns = ["Clim_Rainfall_Mean", "Clim_Temp_Mean", "Clim_Moisture_Mean"]

# Calculate anomalies
df["Rainfall_Anomaly"] = df["Rainfall"] - climatology["Clim_Rainfall_Mean"]
df["Temp_Anomaly"] = df["Temperature"] - climatology["Clim_Temp_Mean"]
df["Soil_Moisture_Anomaly"] = df["Soil_Moisture_Root"] - climatology["Clim_Moisture_Mean"]

print(df[["District", "Year", "Month", "Rainfall_Anomaly", "Soil_Moisture_Anomaly"]].head())

## 5. Crop Physiology Lagged Features

Lags are calculated for Rainfall, Soil Moisture, and historical crop yields.

In [ ]:
# Weather/Moisture Lags
df["Rainfall_Lag_3m"] = df.groupby("District")["Rainfall"].shift(3)
df["Rainfall_Lag_6m"] = df.groupby("District")["Rainfall"].shift(6)
df["Rainfall_Lag_12m"] = df.groupby("District")["Rainfall"].shift(12)
df["Rainfall_Lag_24m"] = df.groupby("District")["Rainfall"].shift(24)

df["Soil_Moisture_Root_Lag_12m"] = df.groupby("District")["Soil_Moisture_Root"].shift(12)
df["Soil_Moisture_Root_Lag_24m"] = df.groupby("District")["Soil_Moisture_Root"].shift(24)

# Yield Lags (using actual Avg_Nuts_Palm target column)
df["Yield_Lag_1y"] = df.groupby("District")["Avg_Nuts_Palm"].shift(12)
df["Yield_Lag_2y"] = df.groupby("District")["Avg_Nuts_Palm"].shift(24)

# Backfill NaNs
lag_cols = [
    "Rainfall_Lag_3m", "Rainfall_Lag_6m", "Rainfall_Lag_12m", "Rainfall_Lag_24m",
    "Soil_Moisture_Root_Lag_12m", "Soil_Moisture_Root_Lag_24m", "Yield_Lag_1y", "Yield_Lag_2y"
]
for col in lag_cols:
    df[col] = df.groupby("District")[col].transform(lambda x: x.bfill())

print("Remaining NaNs in lag columns:", df[lag_cols].isnull().sum().sum())

## 6. Seasonal & Interaction Features

In [ ]:
# Seasonal indices
df["Quarter"] = (df["Month"] - 1) // 3 + 1
df["Month_Sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_Cos"] = np.cos(2 * np.pi * df["Month"] / 12)

# Interaction terms
df["Rain_SoilMoist_Interaction"] = df["Rainfall"] * df["Soil_Moisture_Root"]
df["Temp_Humidity_Interaction"] = df["Temperature"] * df["Humidity"]
df["Light_Temp_Interaction"] = df["Solar_Radiation"] * df["Temperature"]

print(df[["Year", "Month", "Rain_SoilMoist_Interaction", "Temp_Humidity_Interaction", "Light_Temp_Interaction"]].head())

## 7. Export Features Dataset

In [ ]:
out_path = config.PROCESSED_DIR / "master_features.csv"
df.to_csv(out_path, index=False)
print(f"[SUCCESS] Feature engineered dataset saved to {out_path.name}")